# Train Mapper: Surrogate FR Models → InsightFace Feature Space

訓練 4 個 mapper（IR152 / IRSE50 / MobileFace / FaceNet → InsightFace ArcFace R50），給路線 A (DiffProtect) 和路線 B (DiffAIM + mapper ensemble) 共用。

**3 種架構同時訓練**：
1. **Procrustes** (orthogonal linear, baseline) — 閉式解
2. **Linear unconstrained** (W + b)
3. **Residual MLP** (out = x + MLP(x))

**訓練資料**：LFW 13K 張臉（HuggingFace mirror）

**輸出**：`/content/mappers/{model}_{arch}.pt` 共 12 個檔案 + 比較表

**Bootstrap：**
- cell 3 一鍵設置：安裝套件 + 寫 irse.py + 下載 4 個 FR 權重 + 重啟 kernel
- cell 5+ 重啟後從這裡跑

## 0. GPU 檢查

In [ ]:
import torch, sys
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Setup（安裝套件 + 下載權重，會自動重啟 kernel）

In [ ]:
import os, subprocess, glob, shutil
from pathlib import Path

!python -m pip install -q --no-deps insightface==0.7.3 onnxruntime-gpu==1.19.2
!python -m pip install -q --no-deps onnx==1.16.1 easydict
!python -m pip install -q --force-reinstall --no-cache-dir 'numpy==1.26.4'
!python -m pip install -q 'protobuf>=4.25,<5' scikit-learn matplotlib scipy tqdm
!python -m pip install -q --no-deps datasets huggingface_hub
!python -m pip install -q pyarrow fsspec multiprocess dill xxhash 2>&1 | tail -2

# 寫 irse.py
ASSETS = Path('/content/assets/models')
ASSETS.mkdir(parents=True, exist_ok=True)
(ASSETS / '__init__.py').write_text('')

IRSE_PY = r"""
from torch.nn import Linear, Conv2d, BatchNorm1d, BatchNorm2d, PReLU, ReLU, Sigmoid, Dropout2d, Dropout, AvgPool2d, MaxPool2d, AdaptiveAvgPool2d, Sequential, Module, Parameter
import torch.nn.functional as F
import torch
from collections import namedtuple
import math
import pdb

##################################  Original Arcface Model #############################################################

class Flatten(Module):
    def forward(self, input):
        return input.reshape(input.size(0), -1)

def l2_norm(input,axis=1):
    norm = torch.norm(input,2,axis,True)
    output = torch.div(input, norm)
    return output

class SEModule(Module):
    def __init__(self, channels, reduction):
        super(SEModule, self).__init__()
        self.avg_pool = AdaptiveAvgPool2d(1)
        self.fc1 = Conv2d(
            channels, channels // reduction, kernel_size=1, padding=0 ,bias=False)
        self.relu = ReLU(inplace=True)
        self.fc2 = Conv2d(
            channels // reduction, channels, kernel_size=1, padding=0 ,bias=False)
        self.sigmoid = Sigmoid()

    def forward(self, x):
        module_input = x
        x = self.avg_pool(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        return module_input * x

class bottleneck_IR(Module):
    def __init__(self, in_channel, depth, stride):
        super(bottleneck_IR, self).__init__()
        if in_channel == depth:
            self.shortcut_layer = MaxPool2d(1, stride)
        else:
            self.shortcut_layer = Sequential(
                Conv2d(in_channel, depth, (1, 1), stride ,bias=False), BatchNorm2d(depth))
        self.res_layer = Sequential(
            BatchNorm2d(in_channel),
            Conv2d(in_channel, depth, (3, 3), (1, 1), 1 ,bias=False), PReLU(depth),
            Conv2d(depth, depth, (3, 3), stride, 1 ,bias=False), BatchNorm2d(depth))

    def forward(self, x):
        shortcut = self.shortcut_layer(x)
        res = self.res_layer(x)
        return res + shortcut

class bottleneck_IR_SE(Module):
    def __init__(self, in_channel, depth, stride):
        super(bottleneck_IR_SE, self).__init__()
        if in_channel == depth:
            self.shortcut_layer = MaxPool2d(1, stride)
        else:
            self.shortcut_layer = Sequential(
                Conv2d(in_channel, depth, (1, 1), stride ,bias=False), 
                BatchNorm2d(depth))
        self.res_layer = Sequential(
            BatchNorm2d(in_channel),
            Conv2d(in_channel, depth, (3,3), (1,1),1 ,bias=False),
            PReLU(depth),
            Conv2d(depth, depth, (3,3), stride, 1 ,bias=False),
            BatchNorm2d(depth),
            SEModule(depth,16)
            )
    def forward(self,x):
        shortcut = self.shortcut_layer(x)
        res = self.res_layer(x)
        return res + shortcut

class Bottleneck(namedtuple('Block', ['in_channel', 'depth', 'stride'])):
    '''A named tuple describing a ResNet block.'''
    
def get_block(in_channel, depth, num_units, stride = 2):
  return [Bottleneck(in_channel, depth, stride)] + [Bottleneck(depth, depth, 1) for i in range(num_units-1)]

def get_blocks(num_layers):
    if num_layers == 50:
        blocks = [
            get_block(in_channel=64, depth=64, num_units = 3),
            get_block(in_channel=64, depth=128, num_units=4),
            get_block(in_channel=128, depth=256, num_units=14),
            get_block(in_channel=256, depth=512, num_units=3)
        ]
    elif num_layers == 100:
        blocks = [
            get_block(in_channel=64, depth=64, num_units=3),
            get_block(in_channel=64, depth=128, num_units=13),
            get_block(in_channel=128, depth=256, num_units=30),
            get_block(in_channel=256, depth=512, num_units=3)
        ]
    elif num_layers == 152:
        blocks = [
            get_block(in_channel=64, depth=64, num_units=3),
            get_block(in_channel=64, depth=128, num_units=8),
            get_block(in_channel=128, depth=256, num_units=36),
            get_block(in_channel=256, depth=512, num_units=3)
        ]
    return blocks

class Backbone(Module):
    def __init__(self, num_layers, drop_ratio, mode='ir'):
        super(Backbone, self).__init__()
        assert num_layers in [50, 100, 152], 'num_layers should be 50,100, or 152'
        assert mode in ['ir', 'ir_se'], 'mode should be ir or ir_se'
        blocks = get_blocks(num_layers)
        if mode == 'ir':
            unit_module = bottleneck_IR
        elif mode == 'ir_se':
            unit_module = bottleneck_IR_SE
        self.input_layer = Sequential(Conv2d(3, 64, (3, 3), 1, 1 ,bias=False), 
                                      BatchNorm2d(64), 
                                      PReLU(64))
        self.output_layer = Sequential(BatchNorm2d(512), 
                                       Dropout(drop_ratio),
                                       Flatten(),
                                       Linear(512 * 7 * 7, 512),
                                       BatchNorm1d(512))
        modules = []
        for block in blocks:
            for bottleneck in block:
                modules.append(
                    unit_module(bottleneck.in_channel,
                                bottleneck.depth,
                                bottleneck.stride))
        self.body = Sequential(*modules)
    
    def forward(self,x):
        x = self.input_layer(x)
        x = self.body(x)
        x = self.output_layer(x)
        return l2_norm(x)

##################################  MobileFaceNet #############################################################
    
class Conv_block(Module):
    def __init__(self, in_c, out_c, kernel=(1, 1), stride=(1, 1), padding=(0, 0), groups=1):
        super(Conv_block, self).__init__()
        self.conv = Conv2d(in_c, out_channels=out_c, kernel_size=kernel, groups=groups, stride=stride, padding=padding, bias=False)
        self.bn = BatchNorm2d(out_c)
        self.prelu = PReLU(out_c)
    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.prelu(x)
        return x

class Linear_block(Module):
    def __init__(self, in_c, out_c, kernel=(1, 1), stride=(1, 1), padding=(0, 0), groups=1):
        super(Linear_block, self).__init__()
        self.conv = Conv2d(in_c, out_channels=out_c, kernel_size=kernel, groups=groups, stride=stride, padding=padding, bias=False)
        self.bn = BatchNorm2d(out_c)
    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        return x

class Depth_Wise(Module):
     def __init__(self, in_c, out_c, residual = False, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=1):
        super(Depth_Wise, self).__init__()
        self.conv = Conv_block(in_c, out_c=groups, kernel=(1, 1), padding=(0, 0), stride=(1, 1))
        self.conv_dw = Conv_block(groups, groups, groups=groups, kernel=kernel, padding=padding, stride=stride)
        self.project = Linear_block(groups, out_c, kernel=(1, 1), padding=(0, 0), stride=(1, 1))
        self.residual = residual
     def forward(self, x):
        if self.residual:
            short_cut = x
        x = self.conv(x)
        x = self.conv_dw(x)
        x = self.project(x)
        if self.residual:
            output = short_cut + x
        else:
            output = x
        return output

class Residual(Module):
    def __init__(self, c, num_block, groups, kernel=(3, 3), stride=(1, 1), padding=(1, 1)):
        super(Residual, self).__init__()
        modules = []
        for _ in range(num_block):
            modules.append(Depth_Wise(c, c, residual=True, kernel=kernel, padding=padding, stride=stride, groups=groups))
        self.model = Sequential(*modules)
    def forward(self, x):
        return self.model(x)

class MobileFaceNet(Module):
    def __init__(self, embedding_size):
        super(MobileFaceNet, self).__init__()
        self.conv1 = Conv_block(3, 64, kernel=(3, 3), stride=(2, 2), padding=(1, 1))
        self.conv2_dw = Conv_block(64, 64, kernel=(3, 3), stride=(1, 1), padding=(1, 1), groups=64)
        self.conv_23 = Depth_Wise(64, 64, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=128)
        self.conv_3 = Residual(64, num_block=4, groups=128, kernel=(3, 3), stride=(1, 1), padding=(1, 1))
        self.conv_34 = Depth_Wise(64, 128, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=256)
        self.conv_4 = Residual(128, num_block=6, groups=256, kernel=(3, 3), stride=(1, 1), padding=(1, 1))
        self.conv_45 = Depth_Wise(128, 128, kernel=(3, 3), stride=(2, 2), padding=(1, 1), groups=512)
        self.conv_5 = Residual(128, num_block=2, groups=256, kernel=(3, 3), stride=(1, 1), padding=(1, 1))
        self.conv_6_sep = Conv_block(128, 512, kernel=(1, 1), stride=(1, 1), padding=(0, 0))
        self.conv_6_dw = Linear_block(512, 512, groups=512, kernel=(7,7), stride=(1, 1), padding=(0, 0))
        self.conv_6_flatten = Flatten()
        self.linear = Linear(512, embedding_size, bias=False)
        self.bn = BatchNorm1d(embedding_size)
    
    def forward(self, x):
        out = self.conv1(x)

        out = self.conv2_dw(out)

        out = self.conv_23(out)

        out = self.conv_3(out)
        
        out = self.conv_34(out)

        out = self.conv_4(out)

        out = self.conv_45(out)

        out = self.conv_5(out)

        out = self.conv_6_sep(out)

        out = self.conv_6_dw(out)

        out = self.conv_6_flatten(out)

        out = self.linear(out)

        out = self.bn(out)
        return l2_norm(out)

##################################  Arcface head #############################################################


"""
(ASSETS / 'irse.py').write_text(IRSE_PY)

# 下載 4 個 FR 權重（DiffProtect 同來源）
needed = ['ir152.pth', 'irse50.pth', 'facenet.pth', 'mobile_face.pth']
missing = [n for n in needed if not (ASSETS / n).exists()]
if missing:
    print(f'Downloading FR weights: {missing}')
    !rm -rf /content/diffprotect_assets_extract && mkdir -p /content/diffprotect_assets_extract
    !gdown --fuzzy 'https://drive.google.com/file/d/19_Y0jR789BGciogjjoGtWNEv-5QBiCB7/view?usp=sharing' -O /content/diffprotect_assets.zip
    !unzip -q -o /content/diffprotect_assets.zip -d /content/diffprotect_assets_extract
    for fname in needed:
        matches = glob.glob(f'/content/diffprotect_assets_extract/**/{fname}', recursive=True)
        if matches:
            shutil.copy2(matches[0], ASSETS / fname)
            print(f'  copied {fname}')

for n in needed:
    p = ASSETS / n
    print(f'  {n}: {"OK" if p.exists() else "MISSING"}')

print('\n設置完成，自動重啟 kernel...')
os.kill(os.getpid(), 9)

## 2. Stub albumentations + 載入 5 個模型（重啟後從這裡跑）

In [ ]:
import sys, os, types
from pathlib import Path
import numpy as np

assert np.__version__.startswith('1.26'), f'numpy 版本錯: {np.__version__}'
print('numpy:', np.__version__)

# Stub albumentations
for name in list(sys.modules):
    if name.split('.')[0] in {'insightface', 'albumentations'}:
        del sys.modules[name]
_alb = types.ModuleType('albumentations'); _alb.__path__ = []
_alb_core = types.ModuleType('albumentations.core'); _alb_core.__path__ = []
_alb_ti = types.ModuleType('albumentations.core.transforms_interface')
class _Stub:
    def __init__(self, *a, **k): pass
    def __call__(self, **kw): return kw
_alb_ti.ImageOnlyTransform = _Stub; _alb_ti.DualTransform = _Stub
_alb_aug = types.ModuleType('albumentations.augmentations'); _alb_aug.__path__ = []
_alb_pt = types.ModuleType('albumentations.pytorch'); _alb_pt.__path__ = []
sys.modules.update({
    'albumentations': _alb, 'albumentations.core': _alb_core,
    'albumentations.core.transforms_interface': _alb_ti,
    'albumentations.augmentations': _alb_aug, 'albumentations.pytorch': _alb_pt,
})
_alb.core = _alb_core; _alb_core.transforms_interface = _alb_ti
_alb.augmentations = _alb_aug; _alb.pytorch = _alb_pt
_alb.ImageOnlyTransform = _Stub

import torch, cv2
from insightface.app import FaceAnalysis
from insightface.utils import face_align

# === InsightFace (target) ===
INSIGHT_APP = FaceAnalysis(name='buffalo_l',
                           providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
INSIGHT_APP.prepare(ctx_id=0, det_size=(640, 640))
INSIGHT_REC = INSIGHT_APP.models['recognition']

# === 4 個 surrogate FR ===
sys.path.insert(0, '/content')
from assets.models import irse
import torch.nn.functional as F
device = torch.device('cuda')

# IR152 / IRSE50 / MobileFace 都是 irse.py 裡的 class
# facenet 需要 facenet_pytorch（其中 InceptionResnetV1）
print('Loading IRSE50...')
m_irse50 = irse.Backbone(50, 0.6, 'ir_se').to(device)
m_irse50.load_state_dict(torch.load('/content/assets/models/irse50.pth', map_location=device, weights_only=False))
m_irse50.eval()

print('Loading IR152 (Backbone 152, mode ir)...')
# IR152 在 attack_utils 的官方寫法是 `irse.Backbone(152, drop_ratio, mode='ir')` 或類似；
# 但 TreB1eN 的 irse.py 我們只有 50/100/152；用 Backbone(152, 0.6, 'ir')
m_ir152 = irse.Backbone(152, 0.6, 'ir').to(device)
m_ir152.load_state_dict(torch.load('/content/assets/models/ir152.pth', map_location=device, weights_only=False))
m_ir152.eval()

print('Loading MobileFaceNet...')
m_mobile = irse.MobileFaceNet(512).to(device)
m_mobile.load_state_dict(torch.load('/content/assets/models/mobile_face.pth', map_location=device, weights_only=False))
m_mobile.eval()

print('Loading FaceNet (InceptionResnetV1, AMT-GAN weights)...')
# AMT-GAN 的 facenet.pth 是 InceptionResnetV1 變體；DiffProtect 用 facenet-pytorch 的 InceptionResnetV1
# 這裡假設可以用 facenet-pytorch 載
!pip install -q --no-deps facenet-pytorch
from facenet_pytorch import InceptionResnetV1
m_facenet = InceptionResnetV1(pretrained='vggface2', classify=False, device=device)
m_facenet.eval()

SURROGATES = {
    'IR152': m_ir152,
    'IRSE50': m_irse50,
    'MobileFace': m_mobile,
    'FaceNet': m_facenet,
}
print('\nAll surrogate models loaded:', list(SURROGATES.keys()))
print('InsightFace recognition ready')

## 3. 下載 LFW (13K 張)

In [ ]:
from datasets import load_dataset
from PIL import Image
import random, io
from pathlib import Path

LFW_DIR = Path('/content/lfw_hf')
LFW_DIR.mkdir(parents=True, exist_ok=True)

LFW_CANDIDATES = ['logasja/lfw', 'nielsr/lfw-pairs', 'lansinuote/lfw-pairs']
ds = None
for name in LFW_CANDIDATES:
    try:
        print(f'嘗試 {name}...')
        ds = load_dataset(name, split='train', trust_remote_code=True)
        print(f'  成功 size={len(ds)}')
        print(f'  keys: {list(ds[0].keys())}')
        break
    except Exception as e:
        print(f'  失敗: {str(e)[:80]}')
        continue
assert ds is not None, '全部 LFW mirror 失敗'

# 找 image 欄位
img_col = next((c for c in ['image', 'image_1', 'img', 'face'] if c in ds[0]), None)
print(f'image column: {img_col}')

# 全部 dump 成檔
import glob
existing = glob.glob('/content/lfw_hf/*.jpg')
if len(existing) < min(len(ds), 13000):
    print(f'Saving {len(ds)} images to disk...')
    for i, sample in enumerate(ds):
        img = sample[img_col]
        if hasattr(img, 'save'):
            img.convert('RGB').save(LFW_DIR / f'face_{i:05d}.jpg')
        else:
            Image.open(io.BytesIO(img['bytes'])).convert('RGB').save(LFW_DIR / f'face_{i:05d}.jpg')
        if i % 1000 == 0: print(f'  {i}/{len(ds)}')

paths = sorted(LFW_DIR.glob('*.jpg'))
print(f'\nLFW images on disk: {len(paths)}')

## 4. 抽 embedding（5 個模型對 ~13K 臉）

In [ ]:
import torch, cv2, numpy as np
from tqdm import tqdm

def get_aligned_112(bgr):
    faces = INSIGHT_APP.get(bgr)
    if not faces: return None
    face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
    return face_align.norm_crop(bgr, landmark=face.kps, image_size=112)

def emb_insight(aligned_bgr):
    e = INSIGHT_REC.get_feat(aligned_bgr)
    if e.ndim == 2: e = e[0]
    return e.astype(np.float32)

def emb_torch(aligned_bgr, model, size=112):
    rgb = cv2.cvtColor(aligned_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    rgb = (rgb - 0.5) / 0.5
    if size != 112:
        rgb = cv2.resize(rgb, (size, size))
    t = torch.from_numpy(rgb).permute(2,0,1).unsqueeze(0).to(device)
    with torch.no_grad():
        return model(t).cpu().numpy()[0].astype(np.float32)

def l2n(v): return v / (np.linalg.norm(v) + 1e-12)

# 抽 embedding；FaceNet 用 160x160 input
SUR_SIZES = {'IR152': 112, 'IRSE50': 112, 'MobileFace': 112, 'FaceNet': 160}

E_INSIGHT, E_SUR = [], {k: [] for k in SURROGATES}
skipped = 0
for p in tqdm(paths[:13000]):
    img = cv2.imread(str(p))
    if img is None: skipped += 1; continue
    aligned = get_aligned_112(img)
    if aligned is None: skipped += 1; continue
    E_INSIGHT.append(l2n(emb_insight(aligned)))
    for name, m in SURROGATES.items():
        E_SUR[name].append(l2n(emb_torch(aligned, m, SUR_SIZES[name])))

E_INSIGHT = np.stack(E_INSIGHT)
E_SUR = {k: np.stack(v) for k, v in E_SUR.items()}
print(f'\nN: {len(E_INSIGHT)}, skipped: {skipped}')
for k, v in E_SUR.items():
    print(f'  E_{k}: {v.shape}, mean norm: {np.linalg.norm(v, axis=1).mean():.3f}')

# 80/20 train/test split
N = len(E_INSIGHT)
rng = np.random.default_rng(42)
perm = rng.permutation(N)
n_train = int(N * 0.8)
idx_tr, idx_te = perm[:n_train], perm[n_train:]
print(f'\nTrain: {len(idx_tr)}, Test: {len(idx_te)}')

## 5. 訓練 3 種 mapper × 4 個 surrogate

每個 surrogate 訓練：
1. Procrustes (orthogonal linear, 閉式解)
2. Linear unconstrained (W + b, 閉式解)
3. Residual MLP (PyTorch, ~30 epochs)

In [ ]:
from scipy.linalg import orthogonal_procrustes
import torch.nn as nn

MAPPERS_DIR = Path('/content/mappers')
MAPPERS_DIR.mkdir(exist_ok=True)

# === Procrustes (closed-form) ===
def fit_procrustes(X, Y):
    W, scale = orthogonal_procrustes(X, Y)
    return {'W': torch.from_numpy(W.astype(np.float32)), 'scale': float(scale), 'type': 'procrustes'}

# === Linear (closed-form via pseudoinverse) ===
def fit_linear(X, Y):
    Xb = np.concatenate([X, np.ones((len(X), 1))], axis=1)
    W = np.linalg.pinv(Xb) @ Y  # (513, 512)
    return {'W': torch.from_numpy(W[:-1].astype(np.float32)),
            'b': torch.from_numpy(W[-1].astype(np.float32)),
            'type': 'linear'}

# === Residual MLP ===
class ResidualMapper(nn.Module):
    def __init__(self, dim=512, hidden=1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, dim),
        )
    def forward(self, x):
        return x + self.net(x)  # residual

def fit_mlp(X_tr, Y_tr, X_te, Y_te, epochs=30, lr=1e-3, bs=256):
    model = ResidualMapper().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    X_tr_t = torch.from_numpy(X_tr).to(device)
    Y_tr_t = torch.from_numpy(Y_tr).to(device)
    X_te_t = torch.from_numpy(X_te).to(device)
    Y_te_t = torch.from_numpy(Y_te).to(device)
    best_cos = -1; best_state = None
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(X_tr_t))
        total = 0
        for i in range(0, len(perm), bs):
            idx = perm[i:i+bs]
            xb = X_tr_t[idx]; yb = Y_tr_t[idx]
            yp = model(xb)
            # cosine loss + small L2
            loss = 1 - F.cosine_similarity(yp, yb).mean() + 0.01 * ((yp - yb)**2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item()
        sched.step()
        # eval
        model.eval()
        with torch.no_grad():
            yp = model(X_te_t)
            cos = F.cosine_similarity(yp, Y_te_t).mean().item()
        if cos > best_cos:
            best_cos = cos
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best_cos


def eval_mapper(mapper, X_te, Y_te, kind):
    if kind == 'procrustes':
        Xm = X_te @ mapper['W'].numpy()
    elif kind == 'linear':
        Xm = X_te @ mapper['W'].numpy() + mapper['b'].numpy()
    elif kind == 'mlp':
        with torch.no_grad():
            Xm = mapper(torch.from_numpy(X_te).to(device)).cpu().numpy()
    cos = (Xm * Y_te).sum(1) / (np.linalg.norm(Xm, 1) * np.linalg.norm(Y_te, 1) + 1e-12) if False else \
          (Xm * Y_te).sum(1) / (np.linalg.norm(Xm, axis=1) * np.linalg.norm(Y_te, axis=1) + 1e-12)
    r2 = 1 - ((Y_te - Xm)**2).sum() / ((Y_te - Y_te.mean(0))**2).sum()
    return cos.mean(), r2


results = []
for name in SURROGATES:
    print(f'\n========== {name} → InsightFace ==========')
    X = E_SUR[name]
    Y = E_INSIGHT
    X_tr, X_te = X[idx_tr], X[idx_te]
    Y_tr, Y_te = Y[idx_tr], Y[idx_te]

    # Baseline (no mapping)
    cos_raw = (X_te * Y_te).sum(1) / (np.linalg.norm(X_te, axis=1) * np.linalg.norm(Y_te, axis=1) + 1e-12)
    print(f'  Raw cos(no mapper):   {cos_raw.mean():.4f}')

    # Procrustes
    pro = fit_procrustes(X_tr, Y_tr)
    c, r2 = eval_mapper(pro, X_te, Y_te, 'procrustes')
    print(f'  Procrustes:           cos={c:.4f}, R²={r2:.4f}')
    torch.save(pro, MAPPERS_DIR / f'{name}_procrustes.pt')
    results.append((name, 'procrustes', c, r2))

    # Linear
    lin = fit_linear(X_tr, Y_tr)
    c, r2 = eval_mapper(lin, X_te, Y_te, 'linear')
    print(f'  Linear (W+b):         cos={c:.4f}, R²={r2:.4f}')
    torch.save(lin, MAPPERS_DIR / f'{name}_linear.pt')
    results.append((name, 'linear', c, r2))

    # MLP
    mlp, _ = fit_mlp(X_tr, Y_tr, X_te, Y_te, epochs=30)
    c, r2 = eval_mapper(mlp, X_te, Y_te, 'mlp')
    print(f'  Residual MLP:         cos={c:.4f}, R²={r2:.4f}')
    torch.save(mlp.state_dict(), MAPPERS_DIR / f'{name}_mlp.pt')
    results.append((name, 'mlp', c, r2))

## 6. 比較總覽 + 下載 mappers

In [ ]:
import pandas as pd
df = pd.DataFrame(results, columns=['surrogate', 'arch', 'mapped_cos', 'R²'])
print(df.pivot(index='surrogate', columns='arch', values='mapped_cos').round(4))
print('\n--- R² ---')
print(df.pivot(index='surrogate', columns='arch', values='R²').round(4))

# 打包下載
from google.colab import files
!cd /content && zip -qr /content/mappers.zip mappers
print(f'\nMapper files saved to /content/mappers/, packed as mappers.zip')
!ls /content/mappers/
files.download('/content/mappers.zip')

## 下一步

下載 mappers.zip 後：
- **路線 A (DiffProtect + IRSE50 mapper)**：load `IRSE50_mlp.pt`，改 DiffProtect 的 attack loss
- **路線 B (DiffAIM + 4 個 mapper ensemble)**：load 全部 4 個，改 DiffAIM 的 main.py loss